# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad GA1-220501094-01-AA1-EV06**

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Contenido:

- Lección 1: carga de datos (preparación)
- Lección 2: limpieza y estandarización
- Lección 3: filtrado, ordenamiento y agrupación
- Lección 4: combinación de datasets

## Preparación del entorno

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)
print(pd.__version__)

3.0.6


## Lección 1: carga de los cinco archivos

Los CSV usan `;` como separador, por eso se indica `sep=';'`.

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

In [3]:
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Resultado esperado:

```
orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)
```

## Lección 2: limpieza y estandarización

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación. Para cada problema la decisión es **eliminar, corregir o conservar**, según lo que signifique el dato.

### 2.1 Duplicados exactos

In [4]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


**Interpretación:** hay 30 filas (15 pedidos y sus 15 copias). Todas ocurren el miércoles a las 2 a. m., lo que apunta a un fallo del sistema y no a pedidos reales. **Decisión: eliminar.**

In [5]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### 2.2 Tipos de dato con `astype`

In [6]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


Después: 7.2 MB


**Interpretación:** al usar el tipo entero más pequeño que alcanza para cada columna, `orders` baja de 23,0 MB a 7,2 MB. Las columnas con nulos usan `Int16` (con mayúscula), que admite enteros y nulos a la vez.

### 2.3 Nulos: comprobar la causa antes de decidir

In [7]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


**Interpretación:** cada nulo de `days_since_prior_order` es un primer pedido: no existe un pedido anterior. **Decisión: conservar el nulo.** Rellenar con 0 inventaría que el cliente volvió el mismo día.

In [8]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


**Interpretación:** los 1 258 productos sin nombre pertenecen todos al pasillo 100 y al departamento 21 (`missing`). El producto existe, así que no se borra. **Decisión: corregir** con el valor explícito `'Unknown'`.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### 2.4 Texto con los métodos `.str`

In [9]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


**Interpretación:** había 403 nombres con espacios dobles y 16 con espacio invisible (`\xa0`). Tras la limpieza quedan 0 de cada uno, y los nombres quedan en formato título.

### 2.5 Duplicados implícitos

In [10]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


**Interpretación:** quedan 141 nombres repetidos con `product_id` distintos, y casi todos aparecen en pedidos. **Decisión: conservar y documentar.** Borrarlos dejaría pedidos apuntando a productos inexistentes; en los análisis se agrupa por `product_id`, no por nombre.

### 2.6 Limpieza reutilizable con funciones

In [11]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [12]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Verificación superada')

Verificación superada


### Laboratorio guiado 2: pipeline de limpieza con informe de cambios

**Encargo:** construir una secuencia de pasos de limpieza encadenados que parta del archivo crudo, se ejecute con una sola instrucción y deje un informe de lo que cambió en cada paso.

**Paso 1: recargar los datos crudos**

In [13]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


**Paso 2: función que anota el estado de la tabla**

In [14]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

**Paso 3: un paso de limpieza por función**

In [15]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

**Paso 4: encadenar los pasos con `pipe`**

In [16]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

**Paso 5: informe de cambios**

In [17]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos (ver Reto 2, punto 3).

**Paso 6: validar el resultado**

In [18]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


**Conclusiones del laboratorio:** el pipeline produce exactamente las mismas tablas que la limpieza manual (`True True`), deja un informe con cifras de cada paso y termina con una validación automática de las reglas de calidad.

### Reto 2

**Ejercicio 1.** Duplicados exactos en `order_prods`, pares `order_id` + `product_id` repetidos y valores de `reordered`.

In [19]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Respuesta:** no hay filas duplicadas, no se repite ningún par `order_id` + `product_id` y `reordered` solo contiene 0 y 1. `order_prods` no necesita limpieza de duplicados.

**Ejercicio 2.** Nombres duplicados o con espacios en `aisles` y `departments`.

In [20]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    s = df[col]
    print(nombre,
          '| duplicados:', s.duplicated().sum(),
          '| espacios al inicio o final:', (s != s.str.strip()).sum(),
          '| espacios dobles:', s.str.contains('  ', regex=False).sum(),
          '| con mayúsculas:', (s != s.str.lower()).sum())

aisles | duplicados: 0 | espacios al inicio o final: 0 | espacios dobles: 0 | con mayúsculas: 0
departments | duplicados: 0 | espacios al inicio o final: 0 | espacios dobles: 0 | con mayúsculas: 0


**Respuesta:** ninguna de las dos tablas tiene nombres duplicados, espacios sobrantes ni mayúsculas mezcladas. **No necesitan limpieza de texto.**

**Ejercicio 3.** Los 836 nulos de `add_to_cart_order`.

In [21]:
nulos_pos = order_prods[order_prods['add_to_cart_order'].isna()]
tam_pedido = order_prods.groupby('order_id').size()
ids = nulos_pos['order_id'].unique()

print('Nulos:', len(nulos_pos))
print('Pedidos con nulos:', len(ids))
print('Productos por pedido (mínimo y máximo):', tam_pedido[ids].min(), tam_pedido[ids].max())
print('Valor máximo de add_to_cart_order:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos:', (tam_pedido > 64).sum())

Nulos: 836
Pedidos con nulos: 70
Productos por pedido (mínimo y máximo): 65 127
Valor máximo de add_to_cart_order: 64
Pedidos con más de 64 productos: 70


**Respuesta:** los 836 nulos están en 70 pedidos, y cada uno de esos pedidos tiene como mínimo 65 productos (el máximo es 127). El valor más alto que aparece en `add_to_cart_order` es 64, y los 70 pedidos con más de 64 productos son exactamente los que tienen nulos. **Explicación:** el sistema solo numera hasta la posición 64; los productos que pasan de ese límite quedan sin posición registrada. **Decisión: conservar el nulo.** La posición real es desconocida y rellenarla inventaría un dato.

**Ejercicio 4.** Nombres que cambiaron con la estandarización.

In [22]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
con_nombre = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']).sum()
print('Nombres no nulos que cambiaron:', cambiaron)

Nombres no nulos que cambiaron: 6321


**Respuesta:** cambiaron 6 321 nombres: 5 902 solo por el cambio a formato título y 419 por espacios dobles o invisibles (403 + 16).

**Ejercicio 5.** Nombre del día como categoría ordenada y comparación de memoria.

In [23]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
dia_texto = orders['order_dow'].map(dias)
dia_cat = pd.Series(pd.Categorical(dia_texto, categories=list(dias.values()), ordered=True))

print('Texto:    ', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Categoría:', round(dia_cat.memory_usage(deep=True) / 1e6, 2), 'MB')
print(dia_cat.min(), '<', dia_cat.max())

Texto:     7.14 MB
Categoría: 0.48 MB
domingo < sábado


**Respuesta:** como texto ocupa 7,14 MB y como categoría 0,48 MB, unas 15 veces menos, porque la categoría guarda cada nombre una sola vez y usa códigos pequeños. Además, al ser ordenada permite comparar y ordenar los días de domingo a sábado.

**Ejercicio 6.** Filas de `order_prods` de productos `'Unknown'`.

In [24]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas:', len(filas_unknown))
print('Pedidos distintos:', filas_unknown['order_id'].nunique())
print('Porcentaje de las filas:', round(len(filas_unknown) / len(order_prods) * 100, 2), '%')

Filas: 9547
Pedidos distintos: 8254
Porcentaje de las filas: 0.21 %


**Respuesta:** hay 9 547 filas (0,21 % del total) en 8 254 pedidos. **No los eliminaría:** son compras reales, así que borrarlos quitaría productos de pedidos legítimos y alteraría el tamaño de esos carritos y las tasas de recompra. Se conservan marcados como `'Unknown'` y se pueden excluir solo en los análisis donde importe el nombre.

**Ejercicio 7.** `'missing'` como nulo real en una copia de `departments`.

In [25]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.tail(2))
print('Nulos en la copia:', departments_copia['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())

    department_id department
19             20       deli
20             21        NaN
Nulos en la copia: 1
Nulos en la original: 0


**Respuesta:** en la copia, el departamento 21 queda como nulo real y la original no cambia. No conviene hacerlo en la original antes de agrupar porque `groupby` descarta por defecto las filas con clave nula: los productos del departamento `missing` desaparecerían de los conteos sin avisar.

### Documentación de resultados de limpieza (Lección 2)

| Tabla | Problema | Evidencia | Decisión |
| --- | --- | --- | --- |
| `orders` | Filas duplicadas | 15 pedidos repetidos (miércoles, 2 a. m.) | Eliminar |
| `orders` | Tipos sobredimensionados | 23,0 MB | Reducir a 7,2 MB con `astype` |
| `orders` | Nulos en `days_since_prior_order` | 28 817 (todos son primer pedido) | Conservar |
| `products` | Nulos en `product_name` | 1 258 | Rellenar con `'Unknown'` |
| `products` | Espacios dobles e invisibles | 403 y 16 | Estandarizar texto |
| `products` | Nombres repetidos con `product_id` distinto | 141 | Conservar y agrupar por `product_id` |
| `order_prods` | Nulos en `add_to_cart_order` | 836 en 70 pedidos de más de 64 productos | Conservar |
| `order_prods` | Tipos sobredimensionados | 145,4 MB | Reducir a 54,5 MB |
| `aisles`, `departments` | Sin problemas de texto ni duplicados | 0 | Sin cambios |

## Lección 3: filtrado, ordenamiento y agrupación

*(pendiente)*

## Lección 4: combinación de datasets

*(pendiente)*